In [0]:
#Write a pyspark code to obtain a histogram of tweets posted per user in 2022. Output the tweet count per user as the bucket and the number of Twitter users who fall into that bucket.In other words, group the users by the number of tweets they posted in 2022 and count the number of users in each group.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, col, count

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("TweetHistogram") \
    .getOrCreate()

# Sample dataset representing user tweets with timestamps
data = [
    (1, "2022-01-01 10:00:00", "Tweet 1"),
    (1, "2022-02-01 14:00:00", "Tweet 2"),
    (1, "2022-03-01 18:00:00", "Tweet 3"),
    (2, "2022-04-01 09:00:00", "Tweet 1"),
    (2, "2022-05-01 12:00:00", "Tweet 2"),
    (3, "2022-06-01 15:00:00", "Tweet 1"),
    (3, "2022-07-01 16:00:00", "Tweet 2"),
    (3, "2022-08-01 17:00:00", "Tweet 3"),
    (4, "2022-09-01 19:00:00", "Tweet 1"),
    (5, "2022-10-01 20:00:00", "Tweet 1"),
    (5, "2022-11-01 21:00:00", "Tweet 2")
]

# Define schema
schema = ["UserId", "Timestamp", "Tweet"]

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Convert Timestamp to Date and filter tweets posted in 2022
df = df.withColumn("Year", year(col("Timestamp")))

# Filter data for 2022 only
df_2022 = df.filter(col("Year") == 2022)

# Group by user and count the number of tweets each user posted
user_tweet_count = df_2022.groupBy("UserId").agg(count("Tweet").alias("TweetCount"))

# Now, create buckets for the tweet counts
# Here, we will use `when` conditions to define different buckets based on tweet counts

from pyspark.sql.functions import when

# Create buckets for users based on tweet counts
user_tweet_buckets = user_tweet_count.withColumn(
    "TweetBucket", 
    when(col("TweetCount") <= 1, "1 Tweet")
    .when((col("TweetCount") > 1) & (col("TweetCount") <= 3), "2-3 Tweets")
    .when((col("TweetCount") > 3) & (col("TweetCount") <= 5), "4-5 Tweets")
    .otherwise("More than 5 Tweets")
)

# Group by the tweet bucket and count the number of users in each bucket
histogram = user_tweet_buckets.groupBy("TweetBucket").agg(count("UserId").alias("UserCount"))

# Show the result
histogram.show()


+-----------+---------+
|TweetBucket|UserCount|
+-----------+---------+
|    1 Tweet|        1|
| 2-3 Tweets|        4|
+-----------+---------+

